# Extractive Question Answering: Fine-Tuning DistilBERT on Quoref

- Dataset: Quoref, https://huggingface.co/datasets/allenai/quoref
- ~19.4k training questions, ~2.4k validation, over Wikipedia paragraphs
- Unlike SQuAD, Quoref questions specifically require coreferential reasoning: resolving what "he", "she", "it", or "his" refers to before the right answer span can be found
- Full workflow: load, inspect, preprocess, fine-tune, evaluate, demo


## Setup


In [2]:
!pip install -q transformers datasets evaluate accelerate

import numpy as np
import collections
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
import evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00


## 1. Load the Dataset

- Quoref: https://huggingface.co/datasets/allenai/quoref
- Each example has an id, a question, a context paragraph, and answers (text plus a character start position in the context)


In [8]:
data_files = {
    "train": "hf://datasets/allenai/quoref@refs/convert/parquet/default/train/*.parquet",
    "validation": "hf://datasets/allenai/quoref@refs/convert/parquet/default/validation/*.parquet",
}
raw_datasets = load_dataset("parquet", data_files=data_files)
raw_datasets

default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 7.06MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/868k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'context', 'title', 'url', 'answers'],
        num_rows: 19399
    })
    validation: Dataset({
        features: ['id', 'question', 'context', 'title', 'url', 'answers'],
        num_rows: 2418
    })
})

## 2. Inspect the Data

- Look at a few raw examples to see what coreferential reasoning actually looks like here
- Confirm the answer_start offsets line up with the answer text
- Check context and question length


In [10]:
example = raw_datasets["train"][0]
print("Question:", example["question"])
print("Answer:", example["answers"]["text"])
print("Context:", example["context"][:500], "...")
print("Answer Start:", example["answers"]["answer_start"])

Question: What is the first name of the person who doubted it would turn out to be a highly explosive eruption like those that can occur in subduction-zone volcanoes?
Answer: ['Catherine']
Context: The earthquake swarm was noted on October 12, 2007 in the Prince George Citizen by citizen staff, three days after the earthquakes began. Scientists mentioned in the report were seismologist John Cassidy of Natural Resources Canada and volcanologist Catherine Hickson, who was part of the Geological Survey of Canada at the time. At the time of the report, scientists did not know the origin of the swarm. Seismologist John Cassidy stated, "the depth is enough to rule out hydrothermal but it's up in ...
Answer Start: [250]


In [ ]:
# confirm the character offset actually points at the answer text, for a sample of rows
mismatches = 0
for ex in raw_datasets["train"].select(range(1000)):
    start = ex["answers"]["answer_start"][0]
    text = ex["answers"]["text"][0]
    if ex["context"][start : start + len(text)] != text:
        mismatches += 1
print("checked 1000 rows, offset mismatches:", mismatches)

checked 1000 rows, offset mismatches: 30


In [ ]:
context_word_counts = [
    len(ex.split()) for ex in raw_datasets["train"]["context"][:2000]
]
question_word_counts = [
    len(q.split()) for q in raw_datasets["train"]["question"][:2000]
]
print(
    "context length (words), mean:",
    np.mean(context_word_counts),
    "max:",
    np.max(context_word_counts),
)
print(
    "question length (words), mean:",
    np.mean(question_word_counts),
    "max:",
    np.max(question_word_counts),
)

context length (words), mean: 333.983 max: 1142
question length (words), mean: 15.335 max: 48


## 3. Subsample

- Full training set works fine, but a smaller slice keeps fine-tuning time reasonable for a practice run
- Adjust the sizes below depending on how much time is available


In [ ]:
TRAIN_SIZE = 4000
VAL_SIZE = 500

train_dataset_raw = raw_datasets["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
validation_dataset_raw = (
    raw_datasets["validation"].shuffle(seed=42).select(range(VAL_SIZE))
)
print(train_dataset_raw)
print(validation_dataset_raw)

Dataset({
    features: ['id', 'question', 'context', 'title', 'url', 'answers'],
    num_rows: 4000
})
Dataset({
    features: ['id', 'question', 'context', 'title', 'url', 'answers'],
    num_rows: 500
})


## 4. Tokenize and Align Answer Spans

- The model needs a token start position and a token end position, not a character offset
- The tokenizer's offset mapping (which characters each token came from) is what makes that conversion possible
- Long contexts get split into overlapping windows (stride), since a context can run longer than the model's max length


In [19]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
MAX_LENGTH = 384
DOC_STRIDE = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [21]:
def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs


def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])
        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

In [ ]:
train_dataset = train_dataset_raw.map(
    preprocess_training_examples,
    batched=True,
    remove_columns=train_dataset_raw.column_names,
)
validation_dataset = validation_dataset_raw.map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=validation_dataset_raw.column_names,
)
print(train_dataset)
print(validation_dataset)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 6880
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 829
})


## 5. Fine-Tune DistilBERT

- Start from a small pretrained encoder, DistilBERT, and fine-tune its question-answering head
- The head predicts two things per token: how likely it is to be the start of the answer, and how likely it is to be the end


In [24]:
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_CHECKPOINT)

args = TrainingArguments(
    output_dir="distilbert-quoref",
    eval_strategy="no",
    save_strategy="no",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [25]:
trainer.train()

Step,Training Loss


KeyboardInterrupt: 

## 6. Evaluate

- Get start and end logits for every validation window, then turn them back into text answers
- Score with Exact Match and F1 against the true answer spans, the standard metrics for this task


In [ ]:
validation_dataset_for_model = validation_dataset.remove_columns(
    ["example_id", "offset_mapping"]
)
validation_dataset_for_model.set_format("torch")

predictions, _, _ = trainer.predict(validation_dataset_for_model)
start_logits, end_logits = predictions

In [ ]:
def postprocess_predictions(
    start_logits, end_logits, features, examples, n_best=20, max_answer_length=30
):
    example_to_features = collections.defaultdict(list)
    for idx, feature_id in enumerate(features["example_id"]):
        example_to_features[feature_id].append(idx)

    predicted_answers = []
    for example in examples:
        example_id = example["id"]
        context = example["context"]
        answers = []

        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue
                    answers.append(
                        {
                            "text": context[
                                offsets[start_index][0] : offsets[end_index][1]
                            ],
                            "logit_score": start_logit[start_index]
                            + end_logit[end_index],
                        }
                    )

        if answers:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    theoretical_answers = [
        {"id": ex["id"], "answers": ex["answers"]} for ex in examples
    ]
    metric = evaluate.load("squad")
    return (
        metric.compute(predictions=predicted_answers, references=theoretical_answers),
        predicted_answers,
    )


results, predicted_answers = postprocess_predictions(
    start_logits, end_logits, validation_dataset, validation_dataset_raw
)
print(results)

{'exact_match': 15.6, 'f1': 24.853389101095}


In [28]:
# a few predictions next to the true answer, worth a look before trusting the score
for i in range(5):
    example = validation_dataset_raw[i]
    pred = predicted_answers[i]["prediction_text"]
    true = example["answers"]["text"][0]
    print(f"Q: {example['question']}")
    print(f"  predicted: {pred!r}  |  true: {true!r}")

Q: What is the full name of the friend of the man who remembers Native American legends?
  predicted: 'Paul Carlson'  |  true: 'Paul Carlson'
Q: What is the last name of the person who announced that it was lunch time?
  predicted: 'Samuel Allison'  |  true: 'Fermi'
Q: In what year was the park that is between Andapa and Sambava converted to a national park?
  predicted: 'Marojejy'  |  true: '1998'
Q: What did Edith use as the means to maintain the family's social status?
  predicted: 'Britten'  |  true: 'Music'
Q: What are the first names of the people who are hungry?
  predicted: 'Billy Cooper'  |  true: 'Jennie'


## 7. Demo

- Wrap the fine-tuned model into a plain function: a context and a question in, an answer span out
- Tries a couple of validation examples first, then a hand-written coreference example, then an editable cell to try anything


In [ ]:
def answer_question(question, context, model=model, tokenizer=tokenizer):
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=MAX_LENGTH,
    )
    outputs = model(**inputs)
    start_index = outputs.start_logits.argmax()
    end_index = outputs.end_logits.argmax()
    answer_tokens = inputs["input_ids"][0][start_index : end_index + 1]
    return tokenizer.decode(answer_tokens)


for i in range(3):
    example = validation_dataset_raw[i]
    predicted = answer_question(example["question"], example["context"])
    print(f"Q: {example['question']}")
    print(f"  predicted: {predicted!r}  |  true: {example['answers']['text'][0]!r}\n")

RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)

In [ ]:
coref_context = (
    "Sarah handed her sister the keys before she left for the airport. "
    "The car had been parked outside all week."
)
coref_question = "Who left for the airport?"

print(answer_question(coref_question, coref_context))

In [ ]:
your_question = "type your question here"
your_context = "paste or type a short passage here"  # edit both lines and re-run

print(answer_question(your_question, your_context))

## Summary

- Loaded Quoref, a coreference-focused extractive QA dataset, distinct from SQuAD despite the similar shape
- Verified the answer offsets line up with the answer text before trusting any of it
- Converted character-level answer spans into token-level start/end positions using the tokenizer's offset mapping, the trickiest step in the whole pipeline
- Fine-tuned DistilBERT's QA head on a practice-sized subsample
- Evaluated with Exact Match and F1, then wrapped the model into a plain question-in, answer-out function
